## ReadMe

This notebook is used to compute population exposure to a set of hazards for a given country and subnational division (admin1 or admin2). It takes as input population and hazard raster files and admin boundaries. Output is a spreadsheet (.csv and .xlsx) with percentage, per admin subdivision, of people exposed to each one of the three hazards (flood, cyclone and earthquake), both current and future (for flood and cyclone only).

Users only need to indicate country 3-letters iso_code (e.g., 'MOZ' or 'MDG'), admin level ('admin2','admin1') and hazard list. 

This notebook should be executed after country data preparation (country_data_preparation.ipynb)

All file paths and custom parameters are defined on file constants.py

## Custom Functions

In [12]:
def process_all_hazards(admin_df, iso_code, admin_level, hazard_list):
    """
    Loops over all hazard in hazard_list and compute population exposure to each one of them.
    """

    ##Initialise admin dataframe with relevant columns for a given admin level
    admin_pcode_list = {
        'admin0' : [REF_ADMIN_PCODE['admin0']],
        'admin1' : [REF_ADMIN_PCODE['admin0'],REF_ADMIN_PCODE['admin1']],
        'admin2' : [REF_ADMIN_PCODE['admin0'],REF_ADMIN_PCODE['admin1'],REF_ADMIN_PCODE['admin2']],
    }
    
    admin_pcode = REF_ADMIN_PCODE[admin_level]
    current_exposure_df = admin_df[admin_pcode_list[admin_level]].copy()

    ############################
    for hazard in hazard_list:
        exposure_df = process_hazard_exposure(iso_code, hazard, admin_df, admin_pcode)            
        current_exposure_df = current_exposure_df.merge(exposure_df, on = admin_pcode)
        
    ############################

    
    # Create country-specific output path and export data on csv and xlsx formats 
    current_exposure_df = current_exposure_df.round(2)
    free_text = 'population-' + admin_level
    OUTPUT_PATH = ANALYSIS_OUTPUT_PATH.replace('wrl',iso_code.lower()).replace('XXX',free_text)
    #exposure_df.to_csv(OUTPUT_PATH, index=False)    
    current_exposure_df.to_excel(OUTPUT_PATH.replace('.csv','.xlsx'), index=False, sheet_name = 'population')
    print(OUTPUT_PATH)

    return

## Run code

In [13]:
from constants import *
from utils import *

In [ ]:
########################## USER input ####################
iso_code = 'MOZ'
hazard_list = ['flood_current', 'flood_future', 'cyclone_current', 'cyclone_future', 'earthquake']
admin_level_list = ['admin1', 'admin2']
use_gadm_boundaries = True
############################################################


In [ ]:

for admin_level in admin_level_list:
    ##Load admin boundary data
    admin_df = load_admin_data(use_gadm_boundaries, iso_code, admin_level)
    ##Process population exposure
    process_all_hazards(admin_df, iso_code, admin_level, hazard_list)
